# EPS pathway competition -- step-by-step investigation

**Purpose:** investigate whether/how the three EPS pathways (cellulose, colanic acid, PNAG) actually compete for shared precursors when optimized together, rather than each maximized independently with the others ignored.

**Context:** the existing "Total EPS" convention (used in `eps_analysis.ipynb`, `eps_model_KO.ipynb`) sums each pathway's *independently*-maximized flux -- each solved alone, blind to the other two. This is an explicit upper-bound capacity estimate, not a claim about real simultaneous production, and structurally cannot reveal competition either way. A separate earlier test (one joint linear objective, all three combined) collapsed to a degenerate 100%-PNAG / 0%-other corner solution -- real evidence *some* competition exists, but too blunt an approach to show its actual shape, and possibly partly an artifact of how these pathways' lumped stoichiometry was written rather than a clean biological signal.

**This notebook is deliberately a slow, step-by-step workspace, not a finished analysis** -- cells get added one at a time as we work through each question together, not run ahead to a final result.

**Open item, not yet decided:** `eps_ecoli_model_lb.xml` on disk does NOT include the bcsA/pgaB gene-association fixes made in `eps_model_KO.ipynb` (those were applied in-notebook there, never saved back to the XML). This notebook loads the plain saved file as-is below -- whether to carry those fixes over here is an open question, not assumed.

In [1]:
from cobra.io import read_sbml_model
import warnings


model = read_sbml_model('eps_ecoli_model_lb.xml')
BIOMASS_RXN = 'BIOMASS_Ec_iML1515_core_75p37M'
eps_rxns = {'Cellulose': 'EX_cellulose_e', 'Colanic acid': 'EX_colacid_e', 'PNAG': 'EX_puacgam_e'}

print(model)
print(f"Reactions: {len(model.reactions)}, Metabolites: {len(model.metabolites)}, Genes: {len(model.genes)}")

iML1515
Reactions: 2720, Metabolites: 1882, Genes: 1516


## Adding bcsA + pgaB gene associations, saving as a new model file

Same fixes as `eps_model_KO.ipynb` (locus tags verified via BioCyc/EcoCyc there: bcsA=b3533, pgaB=b1023), applied here too so this notebook's model has them. Saved as a **new** file (`eps_ecoli_model_lb_geneassoc.xml`) rather than overwriting `eps_ecoli_model_lb.xml`, so the original saved model is untouched and this version is explicit/opt-in for whatever uses it going forward.

In [2]:
from cobra import Reaction, Metabolite

# bcsA (b3533) -- cellulose synthase catalytic subunit
model.reactions.CELSYNTH.gene_reaction_rule = 'b3533'
model.genes.get_by_id('b3533').name = 'bcsA'

# pgaB (b1023) -- PNAG partial N-deacetylase, required for export
puacgamdeac_p = Metabolite('puacgamdeac_p', formula='C6H12NO4R',
    name='Partially deacetylated poly-beta-1,6-N-acetyl-D-glucosamine', compartment='p')

pgaB_rxn = Reaction('PUACGAMDEAC')
pgaB_rxn.name = 'PNAG partial N-deacetylation (PgaB)'
pgaB_rxn.lower_bound = 0.0
pgaB_rxn.upper_bound = 1000.0
pgaB_rxn.add_metabolites({
    model.metabolites.puacgam_p: -1,
    model.metabolites.h2o_p: -1,
    puacgamdeac_p: 1,
    model.metabolites.ac_p: 1,
})
pgaB_rxn.gene_reaction_rule = 'b1023'
model.add_reactions([pgaB_rxn])
model.genes.get_by_id('b1023').name = 'pgaB'

model.reactions.PUACGAMex.subtract_metabolites({model.metabolites.puacgam_p: -1})
model.reactions.PUACGAMex.add_metabolites({puacgamdeac_p: -1})

print("CELSYNTH genes:", model.reactions.CELSYNTH.gene_reaction_rule)
print("PUACGAMDEAC:", pgaB_rxn.reaction, "| genes:", pgaB_rxn.gene_reaction_rule)
print("PUACGAMex now:", model.reactions.PUACGAMex.reaction)

CELSYNTH genes: b3533
PUACGAMDEAC: h2o_p + puacgam_p --> ac_p + puacgamdeac_p | genes: b1023
PUACGAMex now: puacgamdeac_p --> puacgam_e


In [4]:
from cobra.io import write_sbml_model

write_sbml_model(model, 'eps_ecoli_model_lb_all_genes.xml')
print("Saved: eps_ecoli_model_lb_geneassoc.xml")
print(f"Reactions: {len(model.reactions)}, Metabolites: {len(model.metabolites)}, Genes: {len(model.genes)}")

Saved: eps_ecoli_model_lb_geneassoc.xml
Reactions: 2721, Metabolites: 1883, Genes: 1518
